# Traduction automatique avec un transformeur

Dans ce TP, nous allons mettre en place un modèle de traduction de l'anglais vers le français :

1. utiliser un transformeur **encodeur-décodeur** pré-entraîné, [MarianMT](https://huggingface.co/Helsinki-NLP/opus-mt-en-fr) ;
2. comprendre ce qui se passe pendant l'entraînement (*teacher forcing*) et la génération (recherche en faisceau) ;
3. mesurer la qualité des traductions avec le score BLEU ;
4. affiner (*fine-tuner*) le modèle sur un corpus de traductions de romans, avec une boucle d'entraînement PyTorch.

*Pensez à activer le GPU : Exécution > Modifier le type d'exécution > GPU.*

In [ ]:
!pip install -q datasets sacrebleu sentencepiece

In [ ]:
import random

import sacrebleu
import torch
import tqdm.auto
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Calculs sur : {device}")
torch.manual_seed(0)
random.seed(0)

## Chargement du modèle

Les modèles `opus-mt` de l'université d'Helsinki sont des transformeurs encodeur-décodeur entraînés sur les corpus parallèles [OPUS](https://opus.nlpl.eu/), un modèle par paire de langues.

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

*Inspectez le modèle (`print(model.config)`, `print(model)`) :*

- *combien de couches ont l'encodeur et le décodeur ?*
- *quelle est la dimension du modèle et le nombre de têtes d'attention ?*
- *où se trouve l'attention croisée (*cross-attention*) ? À quoi sert-elle ?*
- *quelle est la taille du vocabulaire ? Quel type de tokenisation est utilisé ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(model.config)
print(model.model.decoder.layers[0])
print(sum(p.numel() for p in model.parameters()) / 1e6, "millions de paramètres")
print(tokenizer.tokenize("The neighbour's cat is extraordinarily lazy."))

- L'encodeur et le décodeur ont chacun 6 couches (`encoder_layers`, `decoder_layers`), de dimension 512 (`d_model`) avec 8 têtes d'attention.
- Chaque couche du décodeur contient une auto-attention **causale** (`self_attn`) puis une attention croisée (`encoder_attn`) : les requêtes viennent du décodeur, les clés et valeurs de la sortie de l'encodeur. C'est ainsi que le décodeur « lit » la phrase source à chaque mot généré.
- Le vocabulaire compte environ 60 000 sous-mots, obtenus avec SentencePiece (le `▁` marque un début de mot).

## Traduire

La méthode `generate` implémente la boucle de décodage. Par défaut, ce modèle utilise une **recherche en faisceau** (*beam search*) : au lieu de ne garder que le mot le plus probable à chaque étape (décodage glouton), elle garde les `num_beams` débuts de traduction les plus probables et choisit à la fin la meilleure traduction complète.

In [ ]:
def translate(sentences: list[str], num_beams: int = 4, batch_size: int = 32) -> list[str]:
  model.eval()
  translations = []
  for i in range(0, len(sentences), batch_size):
    batch = tokenizer(sentences[i:i + batch_size], return_tensors="pt",
                      padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
      generated = model.generate(**batch, num_beams=num_beams, max_length=256)
    translations += tokenizer.batch_decode(generated, skip_special_tokens=True)
  return translations


examples = ["I think, therefore I am.",
            "The weather is lovely today, let's go for a walk.",
            "Could you please send me the report before Friday?"]
for source, target in zip(examples, translate(examples)):
  print(f"{source}\n  → {target}")

*Comparez les traductions obtenues avec `num_beams=1` (décodage glouton) et `num_beams=5`, par exemple sur des phrases ambiguës ou longues. Voyez-vous des différences ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
tricky = ["He saw her duck.",
          "The old man the boats.",
          "Time flies like an arrow; fruit flies like a banana.",
          "Although the committee had initially rejected the proposal, "
          "it eventually approved a revised version after months of negotiation."]
for source, greedy, beam in zip(tricky, translate(tricky, num_beams=1),
                                translate(tricky, num_beams=5)):
  print(f"{source}\n  glouton  → {greedy}\n  faisceau → {beam}")

## Les données : traductions de romans

Nous utilisons [OPUS Books](https://huggingface.co/datasets/Helsinki-NLP/opus_books) : des phrases de romans du domaine public alignées avec leur traduction française. Le style littéraire (dialogues, tournures anciennes) est différent des textes sur lesquels le modèle a surtout été entraîné.

Nous gardons 1 000 paires pour la validation, 1 000 pour le test, et 20 000 pour l'entraînement (pour que le TP reste rapide).

In [ ]:
books = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")
books = books.shuffle(seed=0)
test_pairs = books.select(range(1000))
val_pairs = books.select(range(1000, 2000))
train_pairs = books.select(range(2000, 22000))
for pair in test_pairs.select(range(3))["translation"]:
  print(pair)

## Mesurer la qualité : le score BLEU

Le score [BLEU](https://fr.wikipedia.org/wiki/BLEU_(algorithme)) compare les *n*-grammes (suites de 1 à 4 mots) d'une traduction automatique à ceux d'une traduction de référence. Il va de 0 à 100 ; au-delà de 30, les traductions sont en général compréhensibles, au-delà de 50 de bonne qualité.

C'est une mesure imparfaite (une traduction correcte mais formulée autrement est pénalisée), mais elle permet de comparer des modèles sur un même corpus.

*Codez une fonction `bleu(pairs) -> float` qui traduit les phrases anglaises de `pairs` et calcule le score BLEU avec `sacrebleu.corpus_bleu(traductions, [références])`. Calculez le score du modèle pré-entraîné sur l'ensemble de test.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def bleu(pairs, num_beams: int = 4) -> float:
  sources = [pair["en"] for pair in pairs["translation"]]
  references = [pair["fr"] for pair in pairs["translation"]]
  return sacrebleu.corpus_bleu(translate(sources, num_beams=num_beams), [references]).score


bleu_before = bleu(test_pairs)
print(f"BLEU avant fine-tuning : {bleu_before:.1f}")

## Sous le capot : le *teacher forcing*

Pendant l'entraînement, on ne génère pas mot à mot : on donne au décodeur la **traduction de référence** décalée d'un cran, et on lui demande de prédire chaque mot suivant. Grâce au masque causal, toutes les positions sont prédites en une seule passe.

Le tokeniseur prépare les deux côtés : `text_target` produit les `labels`, et le modèle calcule lui-même la perte d'entropie croisée quand on lui donne ces `labels`.

*Calculez la perte du modèle sur une paire de phrases. Affichez la forme des `logits` : à quoi correspond chaque dimension ?*

In [ ]:
pair = train_pairs[0]["translation"]
batch = tokenizer(pair["en"], text_target=pair["fr"], return_tensors="pt").to(device)
print(batch.keys())
# Votre code ici

### Solution

In [ ]:
with torch.no_grad():
  output = model(**batch)
print(f"Perte : {output.loss.item():.3f}")
print(f"Forme des logits : {tuple(output.logits.shape)}")
print(tokenizer.convert_ids_to_tokens(batch["labels"][0]))

Les logits ont la forme `(batch, longueur de la traduction, taille du vocabulaire)` : pour chaque position de la traduction de référence, un score par sous-mot du vocabulaire.

## Préparation des données d'entraînement

*Codez une fonction `preprocess(examples)` qui tokenise un batch d'exemples du dataset (sources anglaises et cibles françaises, tronquées à 128 tokens), puis appliquez-la avec `train_pairs.map(preprocess, batched=True, remove_columns=...)`.*

*Le `DataCollatorForSeq2Seq` fourni complète ensuite les séquences d'un batch avec du padding. Pourquoi remplace-t-il le padding des `labels` par `-100` ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def preprocess(examples):
  sources = [pair["en"] for pair in examples["translation"]]
  targets = [pair["fr"] for pair in examples["translation"]]
  return tokenizer(sources, text_target=targets, max_length=128, truncation=True)


train_tokenized = train_pairs.map(preprocess, batched=True,
                                  remove_columns=train_pairs.column_names)
print(train_tokenized)

`-100` est l'indice que `cross_entropy` ignore (`ignore_index`) : les positions de padding ne comptent pas dans la perte.

In [ ]:
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
train_loader = DataLoader(train_tokenized, batch_size=32, shuffle=True, collate_fn=collator)
batch = next(iter(train_loader))
print({key: value.shape for key, value in batch.items()})

## Fine-tuning

*Codez la boucle d'entraînement pour une epoch :*

- *optimiseur `AdamW`, learning rate de `5e-5` (petit : on ajuste un modèle déjà bon) ;*
- *un planificateur de learning rate linéaire avec *warmup* (`transformers.get_linear_schedule_with_warmup`, 10 % des pas en warmup) ;*
- *la perte est `model(**batch).loss` ;*
- *affichez la perte moyenne tous les 100 pas.*

*Sur un GPU de Colab, une epoch prend quelques minutes.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
n_steps = len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=n_steps // 10,
                                            num_training_steps=n_steps)

model.train()
running_loss = 0.0
for step, batch in enumerate(tqdm.auto.tqdm(train_loader), 1):
  batch = {key: value.to(device) for key, value in batch.items()}
  optimizer.zero_grad()
  loss = model(**batch).loss
  loss.backward()
  optimizer.step()
  scheduler.step()
  running_loss += loss.item()
  if step % 100 == 0:
    print(f"Pas {step} : perte {running_loss / 100:.3f}")
    running_loss = 0.0

## Évaluation après fine-tuning

*Calculez le score BLEU du modèle affiné sur l'ensemble de test et comparez-le au score avant fine-tuning. Comparez aussi quelques traductions avant/après. Que constatez-vous ?*

In [ ]:
# Votre code ici

### Solution

In [ ]:
bleu_after = bleu(test_pairs)
print(f"BLEU avant : {bleu_before:.1f}, après : {bleu_after:.1f}")

for pair in test_pairs.select(range(5))["translation"]:
  print(f"Source    : {pair['en']}")
  print(f"Référence : {pair['fr']}")
  print(f"Modèle    : {translate([pair['en']])[0]}")
  print("-" * 80)

Le score BLEU progresse après une seule epoch (dans un essai réduit, avec 640 paires d'entraînement seulement, il est passé de 22,4 à 23,5) : le modèle s'adapte au style et au vocabulaire des romans. Les traductions affinées reprennent aussi davantage les tournures de la référence. C'est le principe du transfert d'apprentissage appliqué au texte : partir d'un modèle généraliste et l'ajuster avec peu de données du domaine visé.

Pour aller plus loin :

- vérifiez sur l'ensemble de validation que le modèle ne sur-apprend pas si vous entraînez plus longtemps ;
- sauvegardez le modèle affiné avec `model.save_pretrained("opus-mt-en-fr-books")` et `tokenizer.save_pretrained(...)`, puis rechargez-le avec `from_pretrained`.